# Extraction domain shift — does the validated extractor transfer to 2000–2014?

The extraction layer was validated and calibrated on a 120-case sample stratified over the
**2015–2025** corpus. The 2026-07-07 corpus extension added 397 pre-2015 cases — *outside
the sample frame* (older HUDOC formatting, different drafting conventions). This notebook
turns that stated caveat into an **experiment**:

> Freeze a 40-case stratified sample from 2000–2014, label it with the identical protocol,
> and measure whether (a) the rules extractor's F1 and (b) the **calibration fitted on
> 2015–2025 labels** transfer backward in time.

Either outcome is a finding: transfer = "labelled once, valid for two decades"; failure =
temporal domain shift, quantified. This is also the **reusability question in miniature**:
what does moving the validated component to a new (time-)domain actually cost?

Labels are read only here and in `extraction_validation.ipynb`-style protocol
(single annotator, drafted with LLM assistance, reviewed — stated).

## 1. Stratified pre-2015 sample → label template (frozen once)

In [ ]:
import json
from pathlib import Path
import numpy as np
import pandas as pd

DATA_DIR, REPORT_DIR = Path("../data"), Path("../reports")
TEMPLATE = DATA_DIR / "echr_domain_shift_template.csv"
LABELS   = DATA_DIR / "echr_domain_shift_labels.csv"
SAMPLE_N, SEED = 40, 42

ext = pd.read_parquet(DATA_DIR / "echr_extracted.parquet")
ext["year"] = ext.judgment_date.str[:4].astype(float)
pre = ext[ext.year < 2015].copy()
print(f"pre-2015 cases: {len(pre)} | predicted-positive: {int(pre.alienation_alleged.sum())} "
      f"({pre.alienation_alleged.mean()*100:.1f}%)")

if LABELS.exists():
    print("labels exist — template NOT regenerated (frozen sample)")
elif TEMPLATE.exists():
    print("template exists — NOT regenerated (label it, save as", LABELS.name, ")")
else:
    rng = np.random.default_rng(SEED)
    bands = {
        "pos(>=.5)":     pre[pre.alienation_conf >= 0.5],
        "gray[.25,.5)":  pre[(pre.alienation_conf >= 0.25) & (pre.alienation_conf < 0.5)],
        "low[.05,.25)":  pre[(pre.alienation_conf >= 0.05) & (pre.alienation_conf < 0.25)],
        "none(<.05)":    pre[pre.alienation_conf < 0.05],
    }
    targets = {"pos(>=.5)": 999, "gray[.25,.5)": 999, "low[.05,.25)": 8, "none(<.05)": 12}
    picks = []
    for name, dfb in bands.items():
        k = min(targets[name], len(dfb))
        take = dfb if targets[name] >= len(dfb) else dfb.sample(k, random_state=SEED)
        picks.append(take)
        print(f"  {name:14s}: {len(take)}/{len(dfb)}")
    tmpl = pd.concat(picks).head(SAMPLE_N)
    tmpl = tmpl[["id", "title", "url", "genre", "year", "alienation_conf",
                 "alienation_alleged", "alienation_evidence"]].rename(
        columns={"alienation_alleged": "extractor_guess"})
    tmpl["gold_alienation_alleged"] = ""
    tmpl = tmpl.sample(frac=1, random_state=SEED)
    tmpl.to_csv(TEMPLATE, index=False)
    print(f"wrote frozen template: {len(tmpl)} rows -> {TEMPLATE.name}")

pre-2015 cases: 395 | predicted-positive: 19 (4.8%)
labels exist — template NOT regenerated (frozen sample)


## 2. Transfer evaluation — rules F1 + calibration ECE, pre-2015 vs 2015–2025

In [ ]:
if not LABELS.exists():
    print(f"!! no labels yet: label {TEMPLATE.name}, save as {LABELS.name}, re-run.")
    ev = None
else:
    lab = pd.read_csv(LABELS)
    ev = lab.merge(ext[["id", "alienation_alleged", "alienation_conf",
                        "alienation_conf_cal"]], on="id", how="inner",
                   suffixes=("_tmpl", ""))
    ev["gold"] = pd.to_numeric(ev.gold_alienation_alleged, errors="coerce")
    ev = ev[ev.gold.notna()].copy()
    ev["gold"] = ev.gold.astype(int)
    print(f"labelled pre-2015 cases: {len(ev)} | gold positive rate {ev.gold.mean():.3f}\n")

    from sklearn.metrics import precision_recall_fscore_support, confusion_matrix
    y, yhat = ev.gold.to_numpy(), ev.alienation_alleged.astype(int).to_numpy()
    p, r, f1, _ = precision_recall_fscore_support(y, yhat, average="binary", zero_division=0)
    tn, fp, fn, tp = confusion_matrix(y, yhat, labels=[0, 1]).ravel()
    print("RULES EXTRACTOR TRANSFER (2000-2014 sample vs 2015-2025 reference):")
    print(f"  pre-2015 : P={p:.3f} R={r:.3f} F1={f1:.3f} | TP={tp} FP={fp} FN={fn} TN={tn}")
    print(f"  reference: P=0.543 R=0.475 F1=0.507 (n=120, 2015-2025)\n")

    # calibration transfer: the deployed isotonic was FITTED on 2015-2025 labels only ->
    # evaluating its calibrated confidence on pre-2015 gold = true out-of-domain ECE.
    def ece(conf, yy, n_bins=5):
        bins = np.linspace(0, 1, n_bins + 1)
        idx = np.clip(np.digitize(conf, bins) - 1, 0, n_bins - 1)
        tot = 0.0
        for b in range(n_bins):
            m = idx == b
            if m.sum():
                tot += m.sum() / len(yy) * abs(yy[m].mean() - conf[m].mean())
        return tot
    yf = y.astype(float)
    ece_raw = ece(ev.alienation_conf.to_numpy(float), yf)
    ece_cal = ece(ev.alienation_conf_cal.to_numpy(float), yf)
    print("CALIBRATION TRANSFER (isotonic fitted on 2015-2025, applied to 2000-2014):")
    print(f"  ECE raw = {ece_raw:.3f} | ECE calibrated (out-of-domain) = {ece_cal:.3f}")
    print(f"  reference in-domain (5-fold OOF, 2015-2025): raw 0.179 -> calibrated 0.068")
    metrics = dict(n=len(ev), gold_rate=round(ev.gold.mean(), 3),
                   precision=round(p, 3), recall=round(r, 3), f1=round(f1, 3),
                   ece_raw=round(ece_raw, 3), ece_cal=round(ece_cal, 3))

labelled pre-2015 cases: 40 | gold positive rate 0.525

RULES EXTRACTOR TRANSFER (2000-2014 sample vs 2015-2025 reference):
  pre-2015 : P=0.632 R=0.571 F1=0.600 | TP=12 FP=7 FN=9 TN=12
  reference: P=0.543 R=0.475 F1=0.507 (n=120, 2015-2025)

CALIBRATION TRANSFER (isotonic fitted on 2015-2025, applied to 2000-2014):
  ECE raw = 0.179 | ECE calibrated (out-of-domain) = 0.084
  reference in-domain (5-fold OOF, 2015-2025): raw 0.179 -> calibrated 0.068


## 3. LLM extractor on the same pre-2015 sample (optional, Ollama)

In [ ]:
llm_metrics = None
if ev is not None:
    try:
        import ollama
        ollama.list()
        OLLAMA_OK = True
    except Exception as e:
        OLLAMA_OK = False
        print(f"Ollama unreachable ({e}) — LLM transfer leg skipped")
    if OLLAMA_OK:
        # reuse the deployed prompt/pool/parser from echr_extraction_llm.ipynb + the section
        # splitter/lexicon it depends on from echr_extraction.ipynb (single source of truth)
        import re
        # constants the exec'd cells expect (from echr_extraction_llm config)
        LLM_MODEL, MAX_SENTS, MAX_SENT_CHARS = "llama3.2", 10, 350
        src_ex = json.loads(Path("echr_extraction.ipynb").read_text())
        for marker in ["ALLEGED_THRESHOLD = 0.50", "ECHR_ANCHORS = [", "CLUSTER = re.compile"]:
            cell = next("".join(c["source"]) for c in src_ex["cells"]
                        if c["cell_type"] == "code" and marker in "".join(c["source"]))
            exec(cell, globals())
        src_llm = json.loads(Path("echr_extraction_llm.ipynb").read_text())
        for marker in ["def evidence_pool", "PROMPT = "]:
            cell = next("".join(c["source"]) for c in src_llm["cells"]
                        if c["cell_type"] == "code" and marker in "".join(c["source"]))
            # the pool cell also loads records/labels; strip to the function definitions only
            exec("\n".join(l for l in cell.splitlines()
                           if not l.startswith(("records", "labels", "if LABELED_ONLY",
                                                "    keep", "    records", "print(", "pools",
                                                "n_empty"))), globals())
        CKPT = DATA_DIR / "echr_domain_shift_llm_checkpoint.json"
        done = json.loads(CKPT.read_text()) if CKPT.exists() else {}
        raw_echr = json.loads((DATA_DIR / "echr_parental_alienation.json").read_text())
        by_id = {r.get("itemid"): r for r in raw_echr}
        todo = [i for i in ev.id if i not in done]
        print(f"LLM transfer leg: {len(done)} cached, {len(todo)} to run")
        for n, iid in enumerate(todo, 1):
            pool = evidence_pool(by_id[iid].get("full_text", ""))
            if not pool:
                done[iid] = dict(alleged=False, conf=0.02)
            else:
                out = None
                for _ in range(2):
                    try:
                        out = llm_extract(pool)
                        if out: break
                    except Exception as exn:
                        print("  retry:", exn)
                done[iid] = out or dict(alleged=False, conf=0.5)
            if n % 5 == 0 or n == len(todo):
                CKPT.write_text(json.dumps(done))
                print(f"  {n}/{len(todo)}")
        ev["llm_alleged"] = ev.id.map(lambda i: bool(done[i]["alleged"]))
        from sklearn.metrics import precision_recall_fscore_support
        p2, r2, f2, _ = precision_recall_fscore_support(
            ev.gold, ev.llm_alleged.astype(int), average="binary", zero_division=0)
        print(f"\nLLM (llama3.2) on pre-2015: P={p2:.3f} R={r2:.3f} F1={f2:.3f}")
        print(f"reference (2015-2025, n=120): P=0.585 R=0.600 F1=0.593")
        llm_metrics = dict(precision=round(p2, 3), recall=round(r2, 3), f1=round(f2, 3))

input : echr_parental_alienation.json
output: echr_extracted.parquet | label template: echr_label_template.csv
alleged>= 0.5 | high-conf band 0.7 | sample N=120
section splitter ready | applicant-context sections: {'unparsed', 'FACTS', 'HEADER', 'PROCEDURE'}
alienation extractor ready (lexicon + attribution + negation)
Ollama reachable — model llama3.2
LLM transfer leg: 0 cached, 40 to run
  5/40
  10/40
  15/40
  20/40
  25/40
  30/40
  35/40
  40/40

LLM (llama3.2) on pre-2015: P=0.700 R=0.667 F1=0.683
reference (2015-2025, n=120): P=0.585 R=0.600 F1=0.593


## 4. Report

In [ ]:
if ev is not None:
    lines = ["# Extraction domain-shift report (2000-2014 transfer test)\n\n"]
    lines.append(f"- frozen pre-2015 sample: **{metrics['n']}** cases (stratified by extractor "
                 f"confidence), gold positive rate **{metrics['gold_rate']*100:.0f}%**. "
                 "Protocol identical to the 2015-2025 sample (single annotator, LLM-drafted, "
                 "reviewed; anchoring caveat applies).\n\n")
    lines.append("| leg | 2015-2025 (reference) | 2000-2014 (transfer) |\n|---|---|---|\n")
    lines.append(f"| rules F1 | 0.507 | **{metrics['f1']}** (P {metrics['precision']} / "
                 f"R {metrics['recall']}) |\n")
    if llm_metrics:
        lines.append(f"| LLM (llama3.2) F1 | 0.593 | **{llm_metrics['f1']}** "
                     f"(P {llm_metrics['precision']} / R {llm_metrics['recall']}) |\n")
    lines.append(f"| ECE raw | 0.179 | **{metrics['ece_raw']}** |\n")
    lines.append(f"| ECE calibrated | 0.068 (in-domain OOF) | **{metrics['ece_cal']}** "
                 f"(true out-of-domain: isotonic fitted on 2015-2025 only) |\n\n")
    lines.append("## Reading\n"
                 "- This is the reusability question in miniature: what does moving the "
                 "validated component to a new (time-)domain cost? Comparable F1/ECE = the "
                 "'label once' claim extends two decades back; degradation = temporal domain "
                 "shift, quantified rather than assumed.\n"
                 "- Small n (40) -> wide confidence intervals; this is a transfer *check*, "
                 "not a re-validation.\n")
    (REPORT_DIR / "extraction_domain_shift_report.md").write_text("".join(lines), encoding="utf-8")
    print("wrote", REPORT_DIR / "extraction_domain_shift_report.md")
else:
    print("report skipped — label the template first")

wrote ../reports/extraction_domain_shift_report.md
